In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [3]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import evaluate

print("Loading all 50,000 reviews from IMDb dataset...")
raw_datasets = load_dataset("stanfordnlp/imdb")

# Split the original 25k test set 50/50 into distinct validation and test sets
split_test = raw_datasets["test"].train_test_split(test_size=0.5, seed=42)
raw_datasets["validation"] = split_test["train"]  # 12,500 reviews
raw_datasets["test"] = split_test["test"]          # 12,500 reviews

# 3. Initialize the tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

print("Tokenizing the entire dataset...")
# Full allocation scale mapping profile across all dataset partitions
train_subset = raw_datasets["train"].map(tokenize_function, batched=True)
val_subset = raw_datasets["validation"].map(tokenize_function, batched=True)
test_subset = raw_datasets["test"].map(tokenize_function, batched=True)

# Set up data formatting for PyTorch environments
train_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


# 5. Load the pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 6. Define evaluation metrics (Accuracy and F1 Score)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 7. Define training configurations
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",       # Run validation after every epoch
    save_strategy="epoch",
    load_best_model_at_end=True, # Track the best model based on validation metrics
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    fp16=torch.cuda.is_available()
)

# 8. Initialize the Trainer API
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=val_subset,     # Used for the validation phase during training
    compute_metrics=compute_metrics,
)

# 9. Train the model
trainer.train()

# 10. Run final testing phase
print("\n--- Running evaluation on the unseen test set ---")
test_results = trainer.evaluate(eval_dataset=test_subset)
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")

Loading all 50,000 reviews from IMDb dataset...


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



--- Running evaluation on the unseen test set ---


Test Accuracy: 0.9277


In [4]:
import math
import torch
import torch.nn as nn
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput

# =====================================================================
# 1. DEFINE CUSTOM TRANSFORMER ARCHITECTURE
# =====================================================================
class CustomTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, num_classes=2, max_len=512):
        super(CustomTransformerClassifier, self).__init__()

        # Token embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # Positional Encoding matrix (sine/cosine tensor)
        self.positional_encoding = self._create_positional_encoding(max_len, embed_dim)

        # Transformer Encoder stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head (pools output sequence into class probabilities)
        self.fc = nn.Linear(embed_dim, num_classes)

    def _create_positional_encoding(self, max_len, embed_dim):
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # Shape: [1, max_len, embed_dim]

    def forward(self, x):
        # x shape: [batch_size, seq_len]
        seq_len = x.size(1)

        # Push positional weights to the same device as inputs dynamically
        pe = self.positional_encoding[:, :seq_len, :].to(x.device)

        # Combine token content embeddings with sequence spatial positions
        out = self.embedding(x) + pe

        # Pass sequence through the stack of multi-head self-attention encoders
        out = self.transformer_encoder(out)

        # Mean pooling across the sequence dimension (dim=1)
        out = out.mean(dim=1)

        # Final classification logic
        return self.fc(out)

# =====================================================================
# 2. HUGGING FACE COMPATIBLE TRAINER WRAPPER
# =====================================================================
class HFCompatibleWrapper(nn.Module):
    def __init__(self, custom_model):
        super().__init__()
        self.model = custom_model
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask=None, labels=None):
        logits = self.model(input_ids)
        loss = None
        if labels is not None:
            loss = self.criterion(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)

print("Loading all 50,000 reviews from IMDb dataset...")
raw_datasets = load_dataset("stanfordnlp/imdb")

# Split the original 25k test set 50/50 into distinct validation and test sets
split_test = raw_datasets["test"].train_test_split(test_size=0.5, seed=42)
raw_datasets["validation"] = split_test["train"]  # 12,500 reviews
raw_datasets["test"] = split_test["test"]          # 12,500 reviews

# 3. Initialize the tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

print("Tokenizing the entire dataset...")
# Full allocation scale mapping profile across all dataset partitions
train_subset = raw_datasets["train"].map(tokenize_function, batched=True)
val_subset = raw_datasets["validation"].map(tokenize_function, batched=True)
test_subset = raw_datasets["test"].map(tokenize_function, batched=True)

# Set up data formatting for PyTorch environments
train_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =====================================================================
# 4. MODEL INITIALIZATION & EVALUATION CONFIG
# =====================================================================
raw_custom_model = CustomTransformerClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=128,    # Lightweight dimension choice for faster custom layer computation
    num_heads=4,
    hidden_dim=256,
    num_layers=2
)
model = HFCompatibleWrapper(raw_custom_model)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# =====================================================================
# 5. TRAINING RUN & TEST PHASE EVALUATION
# =====================================================================
training_args = TrainingArguments(
    output_dir="./custom_transformer_results",
    learning_rate=1e-4,          # Custom modules trained from scratch need higher learning rates than pre-trained ones
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",       # Compute validation metrics after every training epoch loop
    save_strategy="epoch",
    load_best_model_at_end=True, # Rollback to the checkpoint with the highest validation accuracy
    metric_for_best_model="accuracy",
    logging_dir="./custom_logs",
    logging_steps=10,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=val_subset,    # Validation split used here to monitor overfitting
    compute_metrics=compute_metrics,
)

print("Starting training loop...")
trainer.train()

print("\n--- Running evaluation on the final, unseen test set ---")
test_results = trainer.evaluate(eval_dataset=test_subset)
print(f"Final Test Accuracy: {test_results['eval_accuracy']:.4f}")


Loading all 50,000 reviews from IMDb dataset...
Tokenizing the entire dataset...


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training loop...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.530774,0.524010,0.737360
2,0.458047,0.465095,0.778640
3,0.428329,0.423339,0.805120
4,0.325558,0.390490,0.826400
5,0.319898,0.392935,0.828480
6,0.311923,0.388733,0.829200
7,0.323593,0.376886,0.837040
8,0.328828,0.379908,0.835120
9,0.263319,0.377529,0.836640
10,0.249793,0.378587,0.838240


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector


--- Running evaluation on the final, unseen test set ---


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Final Test Accuracy: 0.8348


In [5]:
import math
import torch
import torch.nn as nn
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput

# =====================================================================
# 1. OPTIMIZED CUSTOM TRANSFORMER ARCHITECTURE
# =====================================================================
class TunedTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, num_classes=2, max_len=512, dropout=0.2):
        super(TunedTransformerClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = self._create_positional_encoding(max_len, embed_dim)

        # Added dropout to prevent overfitting on small data
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc = nn.Linear(embed_dim, num_classes)

    def _create_positional_encoding(self, max_len, embed_dim):
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def forward(self, x, attention_mask=None):
        seq_len = x.size(1)
        pe = self.positional_encoding[:, :seq_len, :].to(x.device)

        # Apply dropout to combined embeddings
        out = self.dropout(self.embedding(x) + pe)

        # FIX 1: Generate proper padding mask so attention ignores pad tokens
        src_key_padding_mask = None
        if attention_mask is not None:
            # PyTorch expects True/1 for tokens to ignore, Hugging Face gives 1 for tokens to attend to
            src_key_padding_mask = (attention_mask == 0)

        out = self.transformer_encoder(out, src_key_padding_mask=src_key_padding_mask)

        # FIX 2: Pool using the first token ([CLS]) sequence position instead of mean pooling
        out = out[:, 0, :]

        return self.fc(self.dropout(out))

# =====================================================================
# 2. HUGGING FACE COMPATIBLE TRAINER WRAPPER
# =====================================================================
class HFCompatibleWrapper(nn.Module):
    def __init__(self, custom_model):
        super().__init__()
        self.model = custom_model
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask=None, labels=None):
        # Pass the attention mask down to the custom transformer layers
        logits = self.model(input_ids, attention_mask=attention_mask)
        loss = None
        if labels is not None:
            loss = self.criterion(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)

print("Loading all 50,000 reviews from IMDb dataset...")
raw_datasets = load_dataset("stanfordnlp/imdb")

# Split the original 25k test set 50/50 into distinct validation and test sets
split_test = raw_datasets["test"].train_test_split(test_size=0.5, seed=42)
raw_datasets["validation"] = split_test["train"]  # 12,500 reviews
raw_datasets["test"] = split_test["test"]          # 12,500 reviews

# 3. Initialize the tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

print("Tokenizing the entire dataset...")
# Full allocation scale mapping profile across all dataset partitions
train_subset = raw_datasets["train"].map(tokenize_function, batched=True)
val_subset = raw_datasets["validation"].map(tokenize_function, batched=True)
test_subset = raw_datasets["test"].map(tokenize_function, batched=True)

# Set up data formatting for PyTorch environments
train_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =====================================================================
# 4. MODEL INITIALIZATION & EVALUATION CONFIG
# =====================================================================
raw_custom_model = TunedTransformerClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=256,    # Scaled up embed_dim from 128 to 256 for higher capacity
    num_heads=8,      # Scaled up attention heads from 4 to 8
    hidden_dim=512,   # Scaled up feedforward network dimensions
    num_layers=4,     # Scaled up depth layers from 2 to 4
    dropout=0.2       # Clear regularizer boundary
)
model = HFCompatibleWrapper(raw_custom_model)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# =====================================================================
# 5. TRAINING RUN & TEST PHASE EVALUATION
# =====================================================================
training_args = TrainingArguments(
    output_dir="./tuned_transformer_results",
    learning_rate=2e-4,          # Optimized learning rate for scratch transformer convergence
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,          # Increased epochs from 5 to 8 for deeper weight optimization
    weight_decay=0.02,           # Increased weight decay to bound explosive parameter growth
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./tuned_logs",
    logging_steps=50,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=val_subset,
    compute_metrics=compute_metrics,
)

print("Starting optimized training loop...")
trainer.train()

print("\n--- Running evaluation on the final, unseen test set ---")
test_results = trainer.evaluate(eval_dataset=test_subset)
print(f"Final Test Accuracy: {test_results['eval_accuracy']:.4f}")


Loading all 50,000 reviews from IMDb dataset...
Tokenizing the entire dataset...


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting optimized training loop...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.463654,0.433287,0.797840
2,0.380881,0.364666,0.836560
3,0.369858,0.344410,0.852240
4,0.309260,0.346286,0.862880
5,0.282966,0.370975,0.855040
6,0.254743,0.343358,0.860960
7,0.247681,0.354232,0.866640
8,0.244052,0.346092,0.866160
9,0.198421,0.359946,0.858640
10,0.227911,0.357370,0.868400


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type


--- Running evaluation on the final, unseen test set ---


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Final Test Accuracy: 0.8673
